# Step 01: LangSmith 트레이싱 적용

**목표**: SQLAgent와 RAGAgent에 LangSmith 트레이싱을 적용하여 실제 디버깅 환경 구축

**전제 조건**:
- study_01_langsmith.ipynb 완료
- .env에 `LANGSMITH_TRACING=true`, `LANGSMITH_API_KEY` 설정됨

**소요 시간**: 약 20분

---

## 이 노트북에서 배우는 것

| Part | 내용 |
|------|------|
| Part 1 | 환경 확인 |
| Part 2 | SQLAgent에 @traceable 적용 |
| Part 3 | RAGAgent에 @traceable 적용 |
| Part 4 | 실전 디버깅 시나리오 |

## Part 1: 환경 확인

In [7]:
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
load_dotenv()

# 필수 환경 변수 확인
required_vars = [
    "LANGSMITH_TRACING",
    "LANGSMITH_API_KEY",
    "DATABASE_URL",
    "OLLAMA_MODEL",
    "OLLAMA_BASE_URL",
]

print("=== 환경 변수 확인 ===")
all_ok = True
for key in required_vars:
    val = os.environ.get(key, "NOT SET")
    if val == "NOT SET":
        all_ok = False
        print(f"  {key}: NOT SET")
    elif "API_KEY" in key or "PASSWORD" in key or "URL" in key:
        # 민감한 정보는 일부만 표시
        print(f"  {key}: ...{val[-4:]}")
    else:
        print(f"  {key}: {val}")

if all_ok:
    print("\n모든 환경 변수가 설정되었습니다!")
else:
    print("\n.env 파일을 확인하세요!")

=== 환경 변수 확인 ===
  LANGSMITH_TRACING: true
  LANGSMITH_API_KEY: ...05cf
  DATABASE_URL: ...r_db
  OLLAMA_MODEL: qwen3:8b
  OLLAMA_BASE_URL: ...1434

모든 환경 변수가 설정되었습니다!


In [8]:
import requests

# Ollama 서버 상태 확인
ollama_url = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")

try:
    response = requests.get(f"{ollama_url}/api/tags", timeout=5)
    models = response.json().get("models", [])
    print(f"Ollama 서버 연결 성공! ({ollama_url})")
    print(f"사용 가능한 모델: {[m['name'] for m in models]}")
except Exception as e:
    print(f"Ollama 서버에 연결할 수 없습니다: {e}")
    print("터미널에서 'ollama serve' 실행 후 다시 시도하세요.")

Ollama 서버 연결 성공! (http://localhost:11434)
사용 가능한 모델: ['qwen3-hr:latest', 'snowflake-arctic-embed2:latest', 'dengcao/Qwen3-Embedding-8B:Q5_K_M', 'qwen3:8b', 'mistral:7b', 'llama3.1:8b']


## Part 2: SQLAgent 트레이싱

### @traceable 적용 방법

프로젝트의 `SQLAgent`는 LangChain 기반이므로 **자동 트레이싱**됩니다.  
하지만 `@traceable` 래퍼를 추가하면 **더 명확한 계층 구조**를 만들 수 있습니다.

```python
@traceable(name="sql_agent_query", tags=["production", "sql"])
def query_sql_agent(question: str):
    agent = SQLAgent(...)
    return agent.query(question)  # LangChain 호출은 자동으로 child run
```

**LangSmith에서 보이는 구조:**
```
sql_agent_query
  └── ChatOllama (SQL 생성)
  └── SQL 실행
  └── ChatOllama (답변 생성)
```

In [9]:
import sys
from pathlib import Path

# 프로젝트 루트를 path에 추가
project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root))

from langsmith import traceable
from core.agents.sql_agent import SQLAgent
from core.database.connection import DatabaseConnection

print(f"프로젝트 루트: {project_root}")
print("SQLAgent 임포트 완료!")

프로젝트 루트: C:\workspace\enterprise-hr-agent
SQLAgent 임포트 완료!


In [10]:
# 트레이싱이 적용된 SQLAgent 래퍼 함수
@traceable(
    name="sql_agent_query",
    tags=["step_01", "sql", "impl"],
    metadata={"version": "1.0", "purpose": "langsmith_integration"}
)
def query_sql_agent(question: str):
    """
    SQLAgent 쿼리 실행 (LangSmith 트레이싱 포함)
    
    @traceable 데코레이터가 적용되어:
    - 이 함수 호출이 최상위 트레이스로 기록됨
    - 내부 LangChain 호출은 자동으로 child run으로 기록됨
    """
    # 데이터베이스 연결
    db = DatabaseConnection(os.environ["DATABASE_URL"])
    
    # SQLAgent 생성
    agent = SQLAgent(
        db=db,
        model=os.environ.get("OLLAMA_MODEL", "qwen3:8b"),
        provider="ollama",
        base_url=os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
    )
    
    # 쿼리 실행
    return agent.query(question)

print("@traceable이 적용된 query_sql_agent 함수 생성 완료!")

@traceable이 적용된 query_sql_agent 함수 생성 완료!


In [11]:
# SQL Agent 테스트 실행
print("=== SQLAgent 쿼리 실행 ===")
print("질문: 전체 직원 수는 몇 명인가요?\n")

result = query_sql_agent("전체 직원 수는 몇 명인가요?")

print(f"성공: {result['success']}")
print(f"SQL: {result['metadata'].get('sql', 'N/A')}")
print(f"결과: {result['metadata'].get('results', 'N/A')}")
print(f"답변: {result['answer']}")
print()
print("="*50)
print("LangSmith에서 'sql_agent_query' 트레이스를 확인하세요!")
print("https://smith.langchain.com")
print("="*50)

=== SQLAgent 쿼리 실행 ===
질문: 전체 직원 수는 몇 명인가요?

성공: True
SQL: SELECT COUNT(*) FROM employees;
결과: [{'COUNT(*)': 15}]
답변: 전체 직원 수는 15명입니다.

LangSmith에서 'sql_agent_query' 트레이스를 확인하세요!
https://smith.langchain.com


## Part 3: RAGAgent 트레이싱

### RAGAgent 구조

RAGAgent는 다음 단계로 구성됩니다:
1. **문서 검색** (Retriever)
2. **답변 생성** (LLM)

`@traceable`을 적용하면 이 모든 단계가 계층적으로 기록됩니다.

```
rag_agent_query
  └── Retriever (문서 검색)
  └── ChatOllama (답변 생성)
```

In [12]:
from core.agents.rag_agent import RAGAgent

# 트레이싱이 적용된 RAGAgent 래퍼 함수
@traceable(
    name="rag_agent_query",
    tags=["step_01", "rag", "impl"],
    metadata={"version": "1.0", "purpose": "langsmith_integration"}
)
def query_rag_agent(question: str):
    """
    RAGAgent 쿼리 실행 (LangSmith 트레이싱 포함)
    
    @traceable 데코레이터가 적용되어:
    - 이 함수 호출이 최상위 트레이스로 기록됨
    - Retriever, LLM 호출이 child run으로 기록됨
    """
    agent = RAGAgent(
        model=os.environ.get("OLLAMA_MODEL", "qwen3:8b"),
        provider="ollama",
        base_url=os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434"),
        embedding_model=os.environ.get("OLLAMA_EMBEDDING_MODEL", "snowflake-arctic-embed2"),
        top_k=5
    )
    
    return agent.query(question)

print("@traceable이 적용된 query_rag_agent 함수 생성 완료!")

@traceable이 적용된 query_rag_agent 함수 생성 완료!


In [13]:
# RAG Agent 테스트 실행
print("=== RAGAgent 쿼리 실행 ===")
print("질문: 연차 신청 방법을 알려주세요\n")

result = query_rag_agent("연차 신청 방법을 알려주세요")

print(f"성공: {result['success']}")
print(f"검색된 문서 수: {len(result['metadata'].get('source_docs', []))}")
print(f"답변:\n{result['answer'][:500]}..." if len(result['answer']) > 500 else f"답변:\n{result['answer']}")
print()
print("="*50)
print("LangSmith에서 'rag_agent_query' 트레이스를 확인하세요!")
print("https://smith.langchain.com")
print("="*50)

=== RAGAgent 쿼리 실행 ===
질문: 연차 신청 방법을 알려주세요

성공: True
검색된 문서 수: 5
답변:
**답변:**  
연차휴가 신청은 사내 근태 시스템을 통해 신청하며, 사용 예정일로부터 최소 3일 전에 신청을 원칙으로 합니다. 긴급 사용 시 당일 오전 9시 이전 신청도 가능하나, 월 2회를 초과할 수 없습니다.  
- **승인권자:** 3일 이하 연차는 팀장, 4일 이상 연속 사용 시 부서장 승인 필요, 10일 이상 연속 사용 시 본부장 사전 협의 필수.  
- **승인 기한:** 신청 후 24시간 이내 승인 여부 결정, 미응답 시 자동 승인 처리.  
- **긴급 사유:** 업무상 긴급 시 승인 보류 및 대체 일정 협의 가능.

LangSmith에서 'rag_agent_query' 트레이스를 확인하세요!
https://smith.langchain.com


## Part 4: 실전 디버깅 시나리오

### 디버깅이 필요한 상황

실제 프로덕션에서 자주 발생하는 문제들:

1. **SQL Agent 오류**
   - 잘못된 SQL 생성
   - 존재하지 않는 테이블/컬럼 참조
   - 문법 오류

2. **RAG Agent 오류**
   - 관련 없는 문서 검색
   - 할루시네이션

### LangSmith로 디버깅하는 방법

1. 트레이스에서 **입력** 확인 → 프롬프트가 올바른지
2. **child run** 확인 → 어느 단계에서 문제 발생했는지
3. **출력** 확인 → LLM이 무엇을 생성했는지

In [14]:
# 디버깅 시나리오 1: 복잡한 SQL 쿼리
@traceable(
    name="debug_complex_sql",
    tags=["step_01", "debug", "sql"],
    metadata={"scenario": "complex_query"}
)
def debug_complex_sql(question: str):
    """복잡한 SQL 쿼리 디버깅"""
    db = DatabaseConnection(os.environ["DATABASE_URL"])
    agent = SQLAgent(
        db=db,
        model=os.environ.get("OLLAMA_MODEL", "qwen3:8b"),
        provider="ollama",
        base_url=os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
    )
    return agent.query(question)

print("=== 디버깅 시나리오: 복잡한 SQL 쿼리 ===")
complex_question = "부서별 평균 급여와 직원 수를 알려주세요"
print(f"질문: {complex_question}\n")

result = debug_complex_sql(complex_question)

print(f"성공: {result['success']}")
print(f"생성된 SQL:\n{result['metadata'].get('sql', 'N/A')}")
print(f"\n답변: {result['answer'][:300]}..." if len(result['answer']) > 300 else f"\n답변: {result['answer']}")
print()
print("LangSmith에서 'debug_complex_sql' 트레이스의 child run을 확인하세요!")

=== 디버깅 시나리오: 복잡한 SQL 쿼리 ===
질문: 부서별 평균 급여와 직원 수를 알려주세요

성공: True
생성된 SQL:
SELECT departments.name, AVG(salaries.base_salary), COUNT(employees.emp_id)
FROM employees
JOIN departments ON employees.dept_id = departments.dept_id
JOIN salaries ON employees.emp_id = salaries.emp_id
GROUP BY departments.name;

답변: 개발 부서: 평균 5,760,000원, 5명  
영업 부서: 평균 5,420,000원, 5명  
인사 부서: 평균 5,140,000원, 5명

LangSmith에서 'debug_complex_sql' 트레이스의 child run을 확인하세요!


## 완료 체크리스트

### 확인 사항

- [ ] SQLAgent 쿼리가 LangSmith에 트레이스로 기록됨
- [ ] RAGAgent 쿼리가 LangSmith에 트레이스로 기록됨
- [ ] 트레이스에서 입력/출력, 지연시간 확인 가능
- [ ] 태그(`step_01`, `sql`, `rag`)로 필터링 가능

### LangSmith UI에서 확인하기

1. https://smith.langchain.com 접속
2. 프로젝트 선택 (enterprise-hr-project)
3. **Tracing** 탭에서 트레이스 목록 확인
4. 태그 필터: `step_01` 또는 `sql` 또는 `rag`
5. 트레이스 클릭하여 상세 정보 확인

---

## 다음 단계

**step_02_prompt_hub.ipynb**: LangSmith Prompt Hub 활용

- 프롬프트 버전 관리
- A/B 테스트
- 팀 협업

---

**수고하셨습니다!**

이제 SQLAgent와 RAGAgent의 모든 호출을 LangSmith에서 추적할 수 있습니다.  
프로덕션 환경에서 문제가 발생하면 트레이스를 통해 빠르게 디버깅할 수 있습니다.